# NLP Comparison: Regex vs scispaCy vs ClinicalBERT
**Optimization Experiment #3** — Compare latency, throughput, and extraction quality across three NLP backends.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')

_cwd = Path.cwd().resolve()
if os.environ.get("RESULTS_DIR"):
    RESULTS_DIR = Path(os.environ["RESULTS_DIR"]).expanduser().resolve()
elif (_cwd / "data").is_dir():
    RESULTS_DIR = (_cwd / "data" / "results").resolve()
else:
    RESULTS_DIR = (_cwd.parent / "data" / "results").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS = str(RESULTS_DIR / 'nlp_comparison.csv')

In [ ]:
df = pd.read_csv(RESULTS)
df

## 1. Latency per note (ms)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Latency
axes[0].bar(df['method'], df['latency_ms_per_note'], color=sns.color_palette('Set2'))
axes[0].set_title('Latency per note (ms)', fontsize=13)
axes[0].set_ylabel('ms / note')
axes[0].set_xlabel('Method')
for i, v in enumerate(df['latency_ms_per_note']):
    axes[0].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=11)

# Throughput
axes[1].bar(df['method'], df['throughput_notes_per_s'], color=sns.color_palette('Set2'))
axes[1].set_title('Throughput (notes/s)', fontsize=13)
axes[1].set_ylabel('notes / s')
axes[1].set_xlabel('Method')
for i, v in enumerate(df['throughput_notes_per_s']):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=11)

# Avg entities
axes[2].bar(df['method'], df['avg_entities_per_note'], color=sns.color_palette('Set2'))
axes[2].set_title('Avg entities per note', fontsize=13)
axes[2].set_ylabel('entities')
axes[2].set_xlabel('Method')
for i, v in enumerate(df['avg_entities_per_note']):
    axes[2].text(i, v + 0.05, f'{v:.1f}', ha='center', fontsize=11)

plt.suptitle('NLP Backend Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'nlp_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. F1 Score (if gold labels available)

In [ ]:
if df['f1'].max() > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    x = range(len(df))
    width = 0.25
    ax.bar([i - width for i in x], df['precision'], width, label='Precision')
    ax.bar(x, df['recall'], width, label='Recall')
    ax.bar([i + width for i in x], df['f1'], width, label='F1')
    ax.set_xticks(list(x))
    ax.set_xticklabels(df['method'])
    ax.set_ylabel('Score')
    ax.set_title('Precision / Recall / F1 by NLP method')
    ax.legend()
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'nlp_f1.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No gold labels available — run nlp_comparison.py with --gold-csv to populate F1.')

## 3. Latency vs Throughput scatter (trade-off view)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = sns.color_palette('Set2', len(df))
for i, row in df.iterrows():
    ax.scatter(row['latency_ms_per_note'], row['throughput_notes_per_s'],
               s=200, color=colors[i], label=row['method'], zorder=5)
    ax.annotate(row['method'], (row['latency_ms_per_note'], row['throughput_notes_per_s']),
                textcoords='offset points', xytext=(8, 4), fontsize=11)
ax.set_xlabel('Latency (ms/note)')
ax.set_ylabel('Throughput (notes/s)')
ax.set_title('Latency vs Throughput Trade-off')
ax.legend()
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'nlp_tradeoff.png'), dpi=150, bbox_inches='tight')
plt.show()

## Key Observations
- **Regex baseline**: fastest but lowest quality — no semantic understanding
- **scispaCy**: balanced — biomedical NER with UMLS linking at reasonable speed
- **ClinicalBERT**: highest precision but ~10x slower — suitable for offline batch only